# PetPlantr CLIP+DPT Breed Detection Training

**Story 1.4**: Retrain CLIP+DPT pipeline with full set

This notebook demonstrates the training process for the CLIP-based breed detection model.

## Setup and Dependencies

In [ ]:
# Install dependencies
!pip install -r requirements-training.txt

# Set up Python path
import sys
sys.path.append('/Users/medan/Downloads/PetPlantr')

## Import Libraries

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
import logging
from tqdm import tqdm
import matplotlib.pyplot as plt

from src.ai.models.clip_breed import CLIPBreedDetector
from src.ai.training.train_breed_head import BreedDataset, get_transforms

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Load Dataset

In [ ]:
# Dataset configuration
manifest_path = "data/manifest.csv"
image_dir = "data/raw"
batch_size = 32
val_split = 0.2

# Load dataset
print("Loading dataset...")
full_dataset = BreedDataset(manifest_path, image_dir)
print(f"Loaded dataset with {len(full_dataset)} images, {len(full_dataset.breed_to_idx)} breeds")

# Split dataset
train_indices, val_indices = train_test_split(
    range(len(full_dataset)),
    test_size=val_split,
    stratify=[full_dataset.manifest.iloc[i]['breed'] for i in range(len(full_dataset))],
    random_state=42
)

# Create datasets
train_dataset = torch.utils.data.Subset(full_dataset, train_indices)
val_dataset = torch.utils.data.Subset(full_dataset, val_indices)

# Apply transforms
train_dataset.dataset.transform = get_transforms(augment=True)
val_dataset.dataset.transform = get_transforms(augment=False)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

## Initialize Model

In [ ]:
# Model configuration
num_breeds = len(full_dataset.breed_to_idx)
model = CLIPBreedDetector(num_breeds=num_breeds)
model.to(device)

print(f"Model initialized with {num_breeds} breeds")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## Training Configuration

In [ ]:
# Training hyperparameters
epochs = 10
learning_rate = 1e-4

# Optimizer and loss
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

print(f"Training for {epochs} epochs with learning rate {learning_rate}")

## Training Loop

In [ ]:
# Training history
train_losses = []
val_losses = []
train_accs = []
val_accs = []
val_f1s = []

best_f1 = 0.0

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    
    # Train
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels in tqdm(train_loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs['combined_logits'], labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs['combined_logits'].max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    train_loss = total_loss / len(train_loader)
    train_acc = 100. * correct / total
    
    # Validate
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validating"):
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs['combined_logits'], labels)
            
            total_loss += loss.item()
            _, predicted = outputs['combined_logits'].max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    val_loss = total_loss / len(val_loader)
    val_acc = 100. * correct / total
    val_f1 = f1_score(all_labels, all_preds, average='weighted')
    
    # Update learning rate
    scheduler.step()
    
    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    val_f1s.append(val_f1)
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%, Val F1: {val_f1:.4f}")
    
    # Save best model
    if val_f1 > best_f1:
        best_f1 = val_f1
        model.save_model("models/v1.3/best_model", list(full_dataset.breed_to_idx.keys()))
        print(f"New best model saved with F1: {best_f1:.4f}")

print(f"\nTraining completed! Best F1: {best_f1:.4f}")

## Training Results Visualization

In [ ]:
# Plot training curves
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

# Loss
ax1.plot(train_losses, label='Train')
ax1.plot(val_losses, label='Validation')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

# Accuracy
ax2.plot(train_accs, label='Train')
ax2.plot(val_accs, label='Validation')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True)

# F1 Score
ax3.plot(val_f1s)
ax3.set_title('Validation F1 Score')
ax3.set_xlabel('Epoch')
ax3.set_ylabel('F1 Score')
ax3.grid(True)

# Final metrics
ax4.text(0.1, 0.8, f'Final Train Acc: {train_accs[-1]:.2f}%', fontsize=12)
ax4.text(0.1, 0.6, f'Final Val Acc: {val_accs[-1]:.2f}%', fontsize=12)
ax4.text(0.1, 0.4, f'Final Val F1: {val_f1s[-1]:.4f}', fontsize=12)
ax4.text(0.1, 0.2, f'Best Val F1: {best_f1:.4f}', fontsize=12)
ax4.set_xlim(0, 1)
ax4.set_ylim(0, 1)
ax4.axis('off')

plt.tight_layout()
plt.show()

## Model Evaluation

In [ ]:
# Load best model for evaluation
model, breed_names = CLIPBreedDetector.load_model("models/v1.3/best_model")
model.to(device)
model.eval()

# Evaluate on validation set
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Evaluating"):
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        _, predicted = outputs['combined_logits'].max(1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Calculate final metrics
final_accuracy = accuracy_score(all_labels, all_preds)
final_f1 = f1_score(all_labels, all_preds, average='weighted')

print(f"Final Accuracy: {final_accuracy:.4f} ({final_accuracy*100:.2f}%)")
print(f"Final F1 Score: {final_f1:.4f}")

# Check acceptance criteria
accuracy_ok = final_accuracy >= 0.90
f1_ok = final_f1 >= 0.85

print(f"\nAcceptance Criteria:")
print(f"Top-1 Accuracy ≥ 90%: {'✅ PASS' if accuracy_ok else '❌ FAIL'}")
print(f"Breed F1 ≥ 85%: {'✅ PASS' if f1_ok else '❌ FAIL'}")

if accuracy_ok and f1_ok:
    print("\n🎉 Story 1.4 Complete! Model meets all acceptance criteria.")
else:
    print("\n⚠️ Model does not meet acceptance criteria. Additional training may be needed.")

## Save Final Model

In [ ]:
# Save final model
model.save_model("models/v1.3/final_model", list(full_dataset.breed_to_idx.keys()))
print("Final model saved to models/v1.3/final_model")

# Save training configuration
config = {
    "epochs": epochs,
    "batch_size": batch_size,
    "learning_rate": learning_rate,
    "val_split": val_split,
    "final_accuracy": final_accuracy,
    "final_f1": final_f1,
    "num_breeds": num_breeds,
    "device": str(device)
}

import json
with open("models/v1.3/training_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Training configuration saved to models/v1.3/training_config.json")